# Bronze → Silver

Lê as tabelas Delta da camada Bronze, aplica limpeza e normalização, e grava na camada Silver.

**Transformações aplicadas:**
- Remoção da coluna `_id` (artefato do MongoDB)
- Remoção de duplicatas
- Remoção de nulos em colunas de negócio
- Padronização de strings (trim + upper), excluindo metadados
- Adição de `data_hora_silver` para rastreabilidade

> `spark` já está disponível via startup script.

## 1. Configuração

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

BRONZE_PATH = "s3a://bronze/tse"
SILVER_PATH = "s3a://silver/tse"

COLUNAS_METADADOS = {"_id", "data_hora_bronze", "fonte_dados", "data_hora_silver"}

TABELAS = [
    "bens",
    "bens_estatistica",
    "candidatura",
    "cargo",
    "instrucao",
    "municipio",
    "ocupacao",
    "partido",
    "situacao",
    "tipo_bem",
]

print(f"Bronze : {BRONZE_PATH}")
print(f"Silver : {SILVER_PATH}")
print(f"Tabelas: {len(TABELAS)}")

## 2. Funções de limpeza

In [ ]:
def remover_id(df: DataFrame) -> DataFrame:
    if "_id" in df.columns:
        return df.drop("_id")
    return df

def padronizar_strings(df: DataFrame) -> DataFrame:
    for col_name, dtype in df.dtypes:
        if dtype == "string" and col_name not in COLUNAS_METADADOS:
            df = df.withColumn(col_name, F.upper(F.trim(F.col(col_name))))
    return df

def remover_duplicatas(df: DataFrame) -> DataFrame:
    return df.dropDuplicates()

def tratar_nulos(df: DataFrame) -> DataFrame:
    colunas_negocio = [c for c in df.columns if c not in COLUNAS_METADADOS]
    return df.dropna(how="any", subset=colunas_negocio)

def adicionar_timestamp_silver(df: DataFrame) -> DataFrame:
    return df.withColumn("data_hora_silver", F.current_timestamp())

def limpar(df: DataFrame) -> DataFrame:
    df = remover_id(df)
    df = padronizar_strings(df)
    df = remover_duplicatas(df)
    df = tratar_nulos(df)
    df = adicionar_timestamp_silver(df)
    return df

## 3. Processamento Bronze → Silver

In [ ]:
def bronze_para_silver(tabela: str) -> dict:
    origem  = f"{BRONZE_PATH}/{tabela}"
    destino = f"{SILVER_PATH}/{tabela}"

    df = spark.read.format("delta").load(origem)
    total_antes = df.count()

    df = limpar(df)
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(destino)

    total_depois = spark.read.format("delta").load(destino).count()
    removidos = total_antes - total_depois
    print(f"  [{tabela}] {total_antes:,} → {total_depois:,} ({removidos:,} removidos)")
    return {"antes": total_antes, "depois": total_depois}

print("Iniciando processamento Bronze → Silver...\n")
resultados = {}
for tabela in TABELAS:
    try:
        resultados[tabela] = bronze_para_silver(tabela)
    except Exception as e:
        print(f"  [{tabela}] ERRO: {e}")
        resultados[tabela] = {"antes": 0, "depois": 0}

print("\nProcessamento concluído.")

## 4. Validação da camada Silver

In [ ]:
print("=== Validação Silver ===")
print(f"{'Tabela':<25} {'Antes':>10} {'Depois':>10} {'Removidos':>10}")
print("-" * 58)

total_antes = total_depois = 0
for tabela, r in resultados.items():
    removidos = r['antes'] - r['depois']
    print(f"{tabela:<25} {r['antes']:>10,} {r['depois']:>10,} {removidos:>10,}")
    total_antes  += r['antes']
    total_depois += r['depois']

print("-" * 58)
print(f"{'TOTAL':<25} {total_antes:>10,} {total_depois:>10,} {total_antes - total_depois:>10,}")

## 5. Amostra — candidatura (pós-limpeza)

In [ ]:
df_silver = spark.read.format("delta").load(f"{SILVER_PATH}/candidatura")
df_silver.printSchema()
df_silver.show(5, truncate=False)